# Convert ManiSkill HDF5 trajectories to LeRobot

This notebook reads the downloaded PickCube-v1 teleoperation trajectory, inspects its materialized fields, and converts actions plus environment states through the public LePort workflow. The raw demonstration intentionally has no observations. Optional ManiSkill replay can materialize RGB observations before planning, and an optional final step merges two converted episodes.

For action index `i`, `env_states/actors/cube` reads endpoint `i` and `next_env_states/actors/cube` reads endpoint `i + 1`. Both public selectors have `T` frames even though each source leaf contains `T+1` endpoints.

In [1]:
from pathlib import Path
from pprint import pprint

from leport import convert, create_plan, inspect, merge, replay_maniskill, validate
from leport.conversion.plan import save_plan
from leport.sources import EpisodeSelection

# Resolve the project root when running from either the repository or notebooks directory.
project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("Project root not found; start from the leport root or notebooks/")

# Keep the downloaded pair as the stable replay input.
raw_source_path = project_root / "data/maniskill/PickCube-v1-teleop/trajectory.h5"
# Inspect and convert this path; replay replaces it with the generated HDF5 output.
source_path = raw_source_path
output_root = project_root / "outputs"
plan_path = output_root / "maniskill-pickcube-demo.yaml"
target_root = output_root / "maniskill-pickcube-demo"

print("Configured raw source:", raw_source_path)

Configured raw source: /Users/xzhu/Documents/VLA/leport/data/maniskill/PickCube-v1-teleop/trajectory.h5


## Optional: materialize RGB observations with ManiSkill replay

The lightweight adapter can convert the raw action and environment-state fields directly. Set `RUN_REPLAY = True` only when RGB observations are required. Replay uses the separately installed `maniskill-replay` extra, writes a new HDF5/JSON pair beside the raw source, and never changes the original pair.

In [2]:
# Enable only when the simulator should materialize RGB observations.
RUN_REPLAY = False

replay_result = None
if RUN_REPLAY:
    replay_result = replay_maniskill(
        raw_source_path,
        obs_mode="rgb",
        use_env_states=True,
    )
    source_path = replay_result.output_hdf5
    pprint(replay_result.to_dict())
else:
    print("Replay skipped; converting the downloaded raw trajectory.")

Replay skipped; converting the downloaded raw trajectory.


## 1. Inspect the selected source

Inspection validates the HDF5/JSON pair, orders `traj_<id>` groups numerically, projects nested leaves into action-aligned selectors, and reports which materialized fields are image candidates.

In [3]:
# Inspect only the fields materialized in the selected source pair.
if not source_path.is_file():
    raise FileNotFoundError(f"ManiSkill source not found: {source_path}")
inspection = inspect(source_path, adapter="maniskill")
# Keep the committed execution result readable while retaining the important source facts.
pprint(
    {
        "adapter": inspection.adapter,
        "episode_count": len(inspection.episode_ids),
        "first_episode_ids": inspection.episode_ids[:5],
        "total_frames": inspection.total_frames,
        "field_count": len(inspection.fields),
        "field_selectors": [field.selector for field in inspection.fields],
        "diagnostics": inspection.diagnostics,
    }
)

{'adapter': 'maniskill',
 'diagnostics': (),
 'episode_count': 10,
 'field_count': 13,
 'field_selectors': ['actions',
                     'env_states/actors/cube',
                     'env_states/actors/goal_site',
                     'env_states/actors/table-workspace',
                     'env_states/articulations/panda',
                     'next_env_states/actors/cube',
                     'next_env_states/actors/goal_site',
                     'next_env_states/actors/table-workspace',
                     'next_env_states/articulations/panda',
                     'rewards',
                     'success',
                     'terminated',
                     'truncated'],
 'first_episode_ids': ('traj_0', 'traj_1', 'traj_2', 'traj_3', 'traj_4'),
 'total_frames': 1058}


## 2. Define explicit conversion semantics

LePort does not infer FPS, task text, robot type, action meaning, or state composition. The raw workflow concatenates the current Panda articulation and cube environment states. RGB is mapped only after replay has produced and inspection has validated the expected image selector.

In [4]:
# Declare conversion semantics explicitly instead of inferring them from metadata.
fps = 30
task = "pick up the cube"
robot_type = "panda"
action_source = "actions"
state_sources = ("env_states/articulations/panda", "env_states/actors/cube")
action_dtype = "float32"
state_dtype = "float32"
use_videos = False
episode_selection = EpisodeSelection(episode_ids=("traj_0",))

# Map RGB only after replay materializes a valid image field.
rgb_selector = "obs/sensor_data/base_camera/rgb"
rgb_field = inspection.field(rgb_selector)
if RUN_REPLAY:
    if rgb_field is None or not rgb_field.image_candidate:
        raise RuntimeError(f"Replay output does not contain a valid RGB field: {rgb_selector}")
    image_sources = {rgb_selector: "observation.images.base"}
else:
    image_sources = {}

## 3. Create and inspect a conversion plan

Planning reinspects the selected raw or replayed source and verifies selector coverage, transition alignment, dtypes, shapes, target feature names, and semantic inputs before writing target data.

In [5]:
# Build a reviewable plan before writing any target data.
plan = create_plan(
    source_path,
    target_root=target_root,
    repo_id="local/maniskill-pickcube-demo",
    robot_type=robot_type,
    fps=fps,
    task=task,
    action_source=action_source,
    action_dtype=action_dtype,
    state_sources=state_sources,
    state_dtype=state_dtype,
    image_sources=image_sources,
    use_videos=use_videos,
    adapter="maniskill",
    selection=episode_selection,
)
pprint(plan.to_dict())

{'adapter': 'maniskill',
 'features': {'action': {'dtype': 'float32', 'shape': [8]},
              'observation.state': {'dtype': 'float32', 'shape': [44]}},
 'fps': 30,
 'mappings': {'action': {'operation': 'direct', 'sources': ['actions']},
              'observation.state': {'operation': 'concat',
                                    'sources': ['env_states/articulations/panda',
                                                'env_states/actors/cube']}},
 'schema_version': 1,
 'selection': {'episode_ids': ['traj_0'], 'filter_key': None},
 'source': '/Users/xzhu/Documents/VLA/leport/data/maniskill/PickCube-v1-teleop/trajectory.h5',
 'target': {'repo_id': 'local/maniskill-pickcube-demo',
            'robot_type': 'panda',
            'root': '/Users/xzhu/Documents/VLA/leport/outputs/maniskill-pickcube-demo',
            'use_videos': False},
 'task': {'kind': 'static', 'value': 'pick up the cube'}}


## 4. Save the reviewed plan

The YAML file records whether the raw or replayed source was selected and becomes the reproducible contract for conversion and source-aware validation.

In [6]:
# Preserve an existing reviewed plan instead of overwriting it.
output_root.mkdir(parents=True, exist_ok=True)
if plan_path.exists():
    raise FileExistsError(f"Plan already exists: {plan_path}")
save_plan(plan, plan_path)
print("Plan saved:", plan_path)

Plan saved: /Users/xzhu/Documents/VLA/leport/outputs/maniskill-pickcube-demo.yaml


## 5. Convert the trajectory

Conversion streams only the mapped selectors and commits the target after LeRobot reload validation succeeds. If replay was selected, images remain HWC RGB without adapter-side resizing or channel reordering.

In [7]:
# Preserve existing datasets; conversion commits a new target atomically.
if target_root.exists():
    raise FileExistsError(f"Target already exists: {target_root}")
conversion_result = convert(plan_path)
pprint(conversion_result.validation.to_dict())

{'decoded_visual_features': [],
 'episode_lengths': [105],
 'features': {'action': {'dtype': 'float32', 'shape': (8,)},
              'episode_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'frame_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'observation.state': {'dtype': 'float32', 'shape': (44,)},
              'task_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'timestamp': {'dtype': 'float32', 'names': None, 'shape': (1,)}},
 'repo_id': 'local/maniskill-pickcube-demo',
 'root': '/Users/xzhu/Documents/VLA/leport/outputs/maniskill-pickcube-demo',
 'tasks': ['pick up the cube'],
 'total_episodes': 1,
 'total_frames': 105}


## 6. Validate against the source

Providing the reviewed plan checks episode lengths, tasks, planned features, and any decoded visual features against the selected source.

In [8]:
# Validate the completed target against the selected source and plan.
validation_report = validate(target_root, plan=plan_path)
pprint(validation_report.to_dict())

{'decoded_visual_features': [],
 'episode_lengths': [105],
 'features': {'action': {'dtype': 'float32', 'shape': (8,)},
              'episode_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'frame_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'observation.state': {'dtype': 'float32', 'shape': (44,)},
              'task_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'timestamp': {'dtype': 'float32', 'names': None, 'shape': (1,)}},
 'repo_id': 'local/maniskill-pickcube-demo',
 'root': '/Users/xzhu/Documents/VLA/leport/outputs/maniskill-pickcube-demo',
 'tasks': ['pick up the cube'],
 'total_episodes': 1,
 'total_frames': 105}


## 7. Merge converted episodes (optional)

Set `RUN_MERGE = True` to convert `traj_1` from the same selected raw or replayed source, then merge its completed LeRobot dataset with the `traj_0` dataset. Merge never accepts raw HDF5 files directly.

In [9]:
# Keep the second conversion and merge disabled during the default workflow.
RUN_MERGE = False

merge_result = None
if RUN_MERGE:
    second_dataset = output_root / "maniskill-pickcube-demo-b"
    merged_target = output_root / "maniskill-pickcube-demo-merged"
    second_plan = create_plan(
        source_path,
        target_root=second_dataset,
        repo_id="local/maniskill-pickcube-demo-b",
        robot_type=robot_type,
        fps=fps,
        task=task,
        action_source=action_source,
        action_dtype=action_dtype,
        state_sources=state_sources,
        state_dtype=state_dtype,
        image_sources=image_sources,
        use_videos=use_videos,
        adapter="maniskill",
        selection=EpisodeSelection(episode_ids=("traj_1",)),
    )

    if second_dataset.exists():
        raise FileExistsError(f"Target already exists: {second_dataset}")
    second_conversion_result = convert(second_plan)
    pprint(second_conversion_result.validation.to_dict())

    if merged_target.exists():
        raise FileExistsError(f"Merge target already exists: {merged_target}")
    merge_result = merge(
        (target_root, second_dataset),
        target_root=merged_target,
        repo_id="local/maniskill-pickcube-demo-merged",
    )
    pprint(merge_result.to_dict())
else:
    print("Optional merge skipped.")

Optional merge skipped.


## Equivalent CLI commands

The default raw workflow uses only the lightweight `maniskill` extra:

```bash
uv run leport inspect data/maniskill/PickCube-v1-teleop/trajectory.h5 \
  --adapter maniskill --json

uv run leport plan \
  --source data/maniskill/PickCube-v1-teleop/trajectory.h5 \
  --output outputs/maniskill-pickcube-demo.yaml \
  --adapter maniskill \
  --episode traj_0 \
  --target outputs/maniskill-pickcube-demo \
  --repo-id local/maniskill-pickcube-demo \
  --robot-type panda --fps 30 --task "pick up the cube" \
  --action actions --action-dtype float32 \
  --state env_states/articulations/panda \
  --state env_states/actors/cube \
  --state-dtype float32 \
  --no-videos

uv run leport convert --config outputs/maniskill-pickcube-demo.yaml --json
uv run leport validate outputs/maniskill-pickcube-demo \
  --config outputs/maniskill-pickcube-demo.yaml --json

# Optional: convert traj_1 with the same schema before merging.
uv run leport plan \
  --source data/maniskill/PickCube-v1-teleop/trajectory.h5 \
  --output outputs/maniskill-pickcube-demo-b.yaml \
  --adapter maniskill \
  --episode traj_1 \
  --target outputs/maniskill-pickcube-demo-b \
  --repo-id local/maniskill-pickcube-demo-b \
  --robot-type panda --fps 30 --task "pick up the cube" \
  --action actions --action-dtype float32 \
  --state env_states/articulations/panda \
  --state env_states/actors/cube \
  --state-dtype float32 \
  --no-videos

uv run leport convert --config outputs/maniskill-pickcube-demo-b.yaml --json
uv run leport merge \
  outputs/maniskill-pickcube-demo \
  outputs/maniskill-pickcube-demo-b \
  --target outputs/maniskill-pickcube-demo-merged \
  --repo-id local/maniskill-pickcube-demo-merged \
  --json
```

To materialize RGB first, install the replay extra and run the explicit preprocessing command:

```bash
uv sync --extra maniskill --extra maniskill-replay --group dev
uv run leport replay-maniskill data/maniskill/PickCube-v1-teleop/trajectory.h5 \
  --obs-mode rgb --use-env-states --json

uv run leport inspect \
  data/maniskill/PickCube-v1-teleop/trajectory.rgb.pd_joint_pos.physx_cpu.h5 \
  --adapter maniskill --json
```

Use the replayed HDF5 as `--source` and add `--image obs/sensor_data/base_camera/rgb=observation.images.base` to the plan command. Replay and conversion refuse predictable existing outputs; remove or relocate reviewed outputs explicitly before rerunning.